## Environment

In [11]:
!pip install -q gradio timm transformers accelerate sentencepiece pillow


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load shared code

In [12]:
%run ./02_shared.ipynb

c:\Users\THINH\venv\lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## Demo

In [13]:
from pathlib import Path

import gradio as gr
import pandas as pd
import torch


class DemoLoad:
    def __init__(self, config: AConfig) -> None:
        self.config = config
        self.data = DataModule(config)
        self.meta = self.data.meta
        self.builder = ModelBuild(config, self.data.tokenizer)
        self.model_specs = self.read_models()
        self.models = self.load_models()
        self.transform = ImageTransform.eval(config.image_size)

    def read_models(self) -> list[dict[str, str]]:
        result_dir = Path(self.meta["result_dir"])
        compare_path = result_dir / "a_compare.csv"

        if not compare_path.exists():
            raise FileNotFoundError(
                "Run 04_eval_a_only_full_metrics.ipynb before demo."
            )

        table = pd.read_csv(compare_path)

        required_cols = {"config", "decoder", "exact_match"}
        missing = required_cols - set(table.columns)

        if missing:
            raise KeyError(f"a_compare.csv is missing columns: {missing}")

        table = table.sort_values("config")

        return [
            {
                "label": str(row["config"]),
                "decoder": str(row["decoder"]),
                "exact_match": float(row["exact_match"])
            }
            for _, row in table.iterrows()
        ]

    def checkpoint_path(self, decoder: str) -> Path:
        return (
            Path(self.meta["checkpoint_dir"])
            / f"a_{decoder}"
            / "best.pt"
        )

    def load_one_model(self, decoder: str) -> VQAModel:
        model = self.builder.build(decoder)
        ckpt = self.checkpoint_path(decoder)
        state = Checkpoint.load(ckpt, self.config.device)

        if state is None:
            raise FileNotFoundError(f"Missing checkpoint: {ckpt}")

        model.load_state_dict(state["model"])
        model.to(self.config.device)
        model.eval()

        return model

    def load_models(self) -> dict[str, VQAModel]:
        models = {}

        for spec in self.model_specs:
            label = spec["label"]
            decoder = spec["decoder"]
            models[label] = self.load_one_model(decoder)

        return models

    def encode_question(self, question: str) -> dict[str, torch.Tensor]:
        q_data = self.data.tokenizer(
            [str(question)],
            padding="max_length",
            truncation=True,
            max_length=self.config.question_len,
            return_tensors="pt"
        )

        return {
            "question_ids": q_data["input_ids"].to(self.config.device),
            "question_mask": q_data["attention_mask"].to(self.config.device)
        }

    @torch.no_grad()
    def predict_one(
        self,
        model: VQAModel,
        image_tensor: torch.Tensor,
        question_ids: torch.Tensor,
        question_mask: torch.Tensor
    ) -> str:
        generated = model.generate(
            images=image_tensor,
            question_ids=question_ids,
            question_mask=question_mask
        )
        pred = self.data.tokenizer.decode(
            generated[0].detach().cpu().tolist(),
            skip_special_tokens=True
        )

        return pred.strip()

    @torch.no_grad()
    def answer(self, image: object, question: str) -> str:
        if image is None:
            return "Please upload an image."

        image_tensor = self.transform(image.convert("RGB")).unsqueeze(0)
        image_tensor = image_tensor.to(self.config.device)
        q_data = self.encode_question(question)
        lines = []

        for spec in self.model_specs:
            label = spec["label"]
            decoder = spec["decoder"]
            score = spec["exact_match"]
            pred = self.predict_one(
                model=self.models[label],
                image_tensor=image_tensor,
                question_ids=q_data["question_ids"],
                question_mask=q_data["question_mask"]
            )
            lines.append(
                f"{label} ({decoder}): {pred}"
            )

        return "\n".join(lines)

In [ ]:
import gradio as gr


class DemoApp:
    def __init__(self, model: DemoLoad) -> None:
        self.model = model

    def launch(self) -> None:
        app = gr.Interface(
            fn=self.model.answer,
            inputs=[
                gr.Image(type="pil", label="Image"),
                gr.Textbox(label="Question")
            ],
            outputs=gr.Textbox(label="A1 / A2 Answers", lines=5),
            flagging_mode="never"
        )
        app.launch(share=True)

## Launch

In [17]:
config = AConfig()
demo_model = DemoLoad(config)
DemoApp(demo_model).launch()

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://232bb9faac3dcf8d7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
